# Exponent- en mantissatruncatie met vaste schaal ×32 / A32 — honderd epochs

E4M3 is de hoogste MX-bron uit de actuele FP32-gewichten. E2M3 ontstaat door de **hoogste twee exponentbits** te behouden; E2M1 ontstaat vervolgens door de **twee laagste mantissabits** te verwijderen. Alle drie modi gebruiken binnen een minibatch dezelfde E4M3-broncodes. De E4M3-bronschaal blijft behouden voor E4M3; de twee lagere modi gebruiken **E8M0-broncode +5, dus schaal ×32**. De mantissastap naar E2M1 behoudt de aangepaste E2M3-schaal. Dit is een vaste correctie, zonder geleerde schaal. De bron wordt bij volgende minibatches opnieuw uit de actuele gewichten bepaald. Schaaloverflow wordt afgewezen zonder clipping of wraparound.

De FP32-updatemaster is gedeeld; iedere modus heeft een eigen BN-bank die tijdens training normaal meeleert. Activaties blijven A32. Het notebook leest uitsluitend opgeslagen JSON/CSV; het laadt geen dataset, model, checkpoint of optimizer en start geen training.

**Hoofdvergelijking:** het vaste eindpunt **epoch 100** en het beste checkpoint gekozen door E4M3; telkens de drie modi van één en dezelfde epoch. Bij een gecontroleerde vroege stop toont de notebook de laatste beschikbare epoch en meldt hij expliciet dat epoch 100 nog niet is gemeten. Ontbrekende resultaten worden niet voorspeld.


In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # In de runkopie wordt het exacte resultaatpad ingevuld.
SAVE_FIGURES = False  # True: nieuwe analysis_<tijdstip>-map; bestaande resultaten blijven behouden.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

# Gebruik de helper uit déze run, ook als een eerdere notebook dezelfde modulenaam gebruikte.
helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
helper_name = '_mx_triple_scale32_reference100_analysis_helper'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
if spec is None or spec.loader is None:
    raise FileNotFoundError(f'Analysehelper ontbreekt: {helper_path}')
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Epoch 100 en beste checkpoint gekozen door E4M3

Het beste checkpoint is het **eerste strikte maximum van de E4M3-testaccuracy**. Een gelijke score vervangt het eerdere checkpoint niet. E2M3 en E2M1 worden op precies diezelfde epoch gerapporteerd, ook wanneer hun eigen hoogste score elders ligt. Er worden geen afzonderlijke beste epochs tot één resultaat gecombineerd.

De officiële testset wordt volgens het bestaande protocol voor checkpointselectie gebruikt. De beste score is daardoor geen onafhankelijke testschatting na selectie. Het vaste eindpunt epoch 100 blijft apart zichtbaar. Bij een vroeg gestopte run wordt de laatste beschikbare epoch getoond. **Epoch 100 ontbreekt totdat honderd volledige epochs aantoonbaar zijn opgeslagen.** Er wordt geen ontbrekende accuracy geëxtrapoleerd.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'Harde test-CE': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


## Vergelijking met de bestaande ×64-training

De vergelijking toont **epoch 100** en het **gemeenschappelijke checkpoint gekozen op de hoogste E4M3-accuracy per training**. E2M3 en E2M1 komen steeds uit datzelfde checkpoint; hun afzonderlijke beste epochs worden niet gemengd. De volledige ×64-referentie selecteerde E4M3 op epoch 42.

Bij een onderbroken ×32-run stoppen de curves voor beide factoren op dezelfde laatste afgewerkte epoch. Ook het beste ×64-checkpoint wordt dan opnieuw uitsluitend binnen die gedeelde epochrange gekozen. De epoch-100-vergelijking blijft ontbrekend totdat de huidige run compleet en consistent is bevestigd.

De referentiebestanden, trainingsinstellingen en oorspronkelijke modelinitialisatie worden gecontroleerd. Dezelfde seed bewijst geen bitexacte CUDA-herhaalbaarheid; één run per factor geeft geen schatting van variatie tussen seeds.


In [ ]:
reference = ns.load_scale64_reference(run)
display(Markdown(ns.comparison_markdown(run, reference)))
display(ns.comparison_table(run, reference).round(6))
figures['scale_comparison_curves'] = ns.plot_scale_comparison_curves(run, reference)
display(figures['scale_comparison_curves'])
plt.close(figures['scale_comparison_curves'])
figures['scale_comparison_checkpoints'] = ns.plot_scale_comparison_checkpoints(run, reference)
display(figures['scale_comparison_checkpoints'])
plt.close(figures['scale_comparison_checkpoints'])


## Train- en testaccuracy per modus

Alle drie routes worden vanaf een nieuwe initialisatie gezamenlijk getraind. E4M3 gebruikt gewone directe MX-kwantisatie; de twee lagere routes gebruiken de afgesproken bittruncaties van die bron met de vaste gedeelde schaaloffset +5. De testfase gebruikt **dezelfde truncatieroutes** als de training.

Trainaccuracy is tijdens de updates verzameld. Testaccuracy wordt na iedere epoch met bevroren evaluatiestaat gemeten, op dezelfde volledige testset voor alle modi. BN-statistieken worden tijdens training per modus geleerd; zij worden tijdens de testfase niet aangepast.


In [ ]:
figures['accuracy_curves'] = ns.plot_learning(run)
display(figures['accuracy_curves'])
plt.close(figures['accuracy_curves'])


## Blijft het netwerk verschillende cijfers voorspellen?

De eerdere post-hocproeven lieten bij sommige routes sterk vertekende of constante voorspellingen zien. Daarom wordt hier per testepoch en per modus bijgehouden hoe de voorspellingen over de tien cijfers verdeeld zijn. Een constante voorspelling is direct zichtbaar als **één gebruikte klasse en 100% concentratie**. De tabel toont ook hoe vaak het dominante cijfer werkelijk in de testset voorkomt.

Dit zijn geaggregeerde tellingen; deze notebook doet geen nieuwe inferentie en leidt er geen per-beeldvergelijking uit af.


In [ ]:
display(ns.prediction_table(run, selected_only=True))
figures['prediction_diversity'] = ns.plot_prediction_diversity(run)
display(figures['prediction_diversity'])
plt.close(figures['prediction_diversity'])


## Losses: trainingsdoelen onderscheiden van harde test-CE

De trainpass van E4M3 gebruikt harde CE van labels. Daarna leert E2M3 met soft-target CE van de **detached E4M3-softmax**; E2M1 leert met soft-target CE van de **detached E2M3-softmax**. Dit is recursieve distillatie met T=1. De drie gradients worden gesommeerd zonder deling door drie, waarna één Adam-update volgt; er is geen extra FP32-teacherforward.

De drie trainobjectieven staan in afzonderlijke panelen omdat hun targets verschillen. Soft CE is geen harde label-CE en wordt hier ook niet als KL-divergentie aangeduid. **Testloss is voor alle drie modi harde CE van dezelfde labels** en staat daarom samen in één vergelijkbaar paneel.


In [ ]:
if not run.metrics.empty:
    objective_columns = [key for key in ('epoch', 'phase', 'weight_format', 'objective_type', 'objective_loss') if key in run.metrics]
    display(run.metrics[objective_columns].round({'objective_loss': 8}))
figures['objective_losses'] = ns.plot_objectives(run)
display(figures['objective_losses'])
plt.close(figures['objective_losses'])


In [ ]:
display(Markdown('## Gemeten trainingsduur en eventuele projectie bij een vroege stop'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nDe opgeslagen pieken betreffen de PyTorch CUDA-allocator. Zij meten niet alle GPU-processen of compacte native MX-uitvoering. Ontbrekende waarden blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabellen rapporteren bestaande controles uit de run. Het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates\n\nDe mastergewichten zijn dezelfde gedeelde tensors. De herhaalde masterkolommen per modus mogen **niet bij elkaar worden opgeteld**; code- en schaalwijzigingen behoren wel bij het betreffende formaat.'))
display(ns.weight_update_table(run))
display(Markdown('### Eén opgeslagen E4M3-bron en twee afgeleide exports per geselecteerd checkpoint'))
display(ns.export_table(run))


## Interpretatie en grenzen

Deze training onderzoekt of gezamenlijk trainen met de **naïeve exponentbitbewerking, vaste schaal ×32 en daaropvolgende mantissabitshift** de drie uitvoermodi laat leren. De lagere modi gebruiken geen directe E2-encoder of afzonderlijke E2-schaalkeuze. De +5-offset geldt tegenover de actuele E4M3-bronschaal; de schaal is niet voor de volledige training vastgezet op haar beginwaarde. De identity-STE geldt voor de volledige forwardbewerking naar G(W), zonder extra factor32 in de backward. De BN-banken leren tijdens de gewone training mee. Dit is vaste schaal ×32; de nearest-even omzetting van route C is geen onderdeel van deze training.

Een lage accuracy of voorspelling van één vaste klasse blijft een onderzoeksuitkomst zolang de technische controles slagen. Een dalende soft-target trainingsloss bewijst op zichzelf geen verbeterde beeldherkenning. Bekijk daarom steeds harde test-CE, testaccuracy én de voorspellingstellingen.

Deze proef gebruikt één seed. Vergelijkingen met de directe referentie, de eerdere naïeve truncatietraining of post-hocproeven betreffen apart getrainde gewichten en bijbehorende BN-banken; zij isoleren niet het afzonderlijke aandeel van masteraanpassing en BN-aanpassing. Deze notebook presenteert opgeslagen metingen van deze training en de gecontroleerde historische ×64-referentie. De beste checkpointscore op de officiële testset en de score op het vaste eindpunt beantwoorden verschillende selectievragen.

De bronexport bevat de E4M3-codes, E8M0-schalen en benodigde overige modelstaat. De afgeleide exports worden uit de opgeslagen en herladen bron gemaakt; de zes MX-updatemasters zijn daarin afwezig. De exporttabel maakt zichtbaar welke reconstructiecontroles zijn uitgevoerd. De drie BN-banken en FP32-uitzonderingslagen blijven benodigde modelstaat.

A32 en FP32 beschrijven tensor- en masterdtypes. De opgeslagen CUDA-backendprecisie staat apart in de protocoltabel; cuDNN kan intern TF32 gebruiken. Dit is een MX-emulator, geen meting van native MX-snelheid of energie.

`SAVE_FIGURES = True` schrijft de acht figuren en een analysemanifest naar een **nieuwe** map. Standaard staat deze schakelaar uit, ook in de runkopie. JSON, CSV, checkpoints en eerdere analyses blijven ongewijzigd.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')
print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [str(path.relative_to(run.path)) if path.is_relative_to(run.path) else str(path) for path in run.input_files]}))
